In [1]:
!pip install -U gradio google-generativeai python-dotenv

In [2]:
import os
from dotenv import load_dotenv
from typing import Literal, Tuple
import gradio as gr
import google.generativeai as genai

In [3]:
# Load API Key
load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

genai.configure(api_key=GOOGLE_API_KEY)
model = genai.GenerativeModel("gemini-1.5-flash")

In [4]:
def _call_model(prompt: str, model_name: str) -> str:
    m = genai.GenerativeModel(model_name)
    resp = m.generate_content(prompt)
    return resp.text


In [5]:
SYSTEM_INSTRUCTIONS = (
    "You are an Email Rewriter assistant. You rewrite emails in the requested tone, "
    "preserving the original intent and facts, improving clarity, grammar, and flow. "
    "Do NOT invent commitments, dates, or facts. Use clear, concise language."
)

TONE_PHRASES = {
    "Formal": "formal, professional, and courteous",
    "Casual": "casual, friendly, and approachable",
    "Persuasive": "persuasive and benefit-focused, with confident but polite language",
    "Concise": "concise and to the point while remaining polite",
    "Polite": "polite and empathetic without being overly wordy",
}

In [12]:
def build_prompt(raw_email: str, tone: str, word_limit: int, include_subject: bool, extra_style: str) -> str:
    limit_clause = (
        f"Strictly limit the rewritten email to under {word_limit} words. "
        if word_limit and word_limit > 0 else ""
    )
    subject_clause = (
        "Generate a clear and professional subject line. Place it at the top in the format:\nSubject: <text>\n\n"
        if include_subject else ""
    )
    extra_clause = (
        f"Additionally, apply this style guidance: {extra_style}. "
        if extra_style else ""
    )
    format_clause = "Subject: <text>\n\n<email body>" if include_subject else "<email body only>"

    prompt = f"""
{SYSTEM_INSTRUCTIONS}

Your task:
- Rewrite the email below in a {TONE_PHRASES.get(tone, tone)} tone.
- Preserve the meaning, facts, and intent. Do not invent new details.
- Improve clarity, grammar, and flow.
- {limit_clause}{extra_clause}
- {subject_clause if include_subject else "Do not generate a subject line."}

Format the response exactly as follows:
{format_clause}

Original email:
{raw_email.strip()}
"""
    return prompt

In [13]:
def rewrite(
    email_text: str,
    tone: Literal["Formal", "Casual", "Persuasive", "Concise", "Polite"],
    word_limit: int,
    include_subject: bool,
    extra_style: str,
    model_name: str,
) -> Tuple[str, str]:
    if not email_text.strip():
        return "", "Please paste an email to rewrite."

    try:
        prompt = build_prompt(email_text, tone, word_limit, include_subject, extra_style)
        output = _call_model(prompt, model_name or DEFAULT_MODEL).strip()
        if not output:
            return "", "No text returned by the model. Try again or switch models."
        return output, ""
    except Exception as e:
        return "", f"Error: {e}"


In [24]:
demo = gr.Blocks(theme=gr.themes.Soft(), css=".wrap {max-width: 920px; margin: 0 auto}")

with demo:
    :
    gr.Markdown(
        """
        <h1 style='text-align: center;'>✉️ ComposeAI</h1>
        <p style='text-align: center; font-size: 18px; color: gray;'>
        AI-powered email composition
        </p>
        """,
        unsafe_allow_html=True,
    )


    with gr.Row():
        with gr.Column(scale=2):
            email_in = gr.Textbox(
                label="Your rough email (required)",
                placeholder="Paste your email here...",
                lines=12,
            )
        with gr.Column(scale=1):
            tone = gr.Dropdown(
                ["Formal", "Casual", "Persuasive", "Concise", "Polite"],
                value="Formal",
                label="Tone (required)",
            )
            word_limit = gr.Slider(0, 400, value=0, step=10, label="Word limit (optional, 0 = no limit)")
            include_subject = gr.Checkbox(True, label="Suggest a subject line")
            extra_style = gr.Textbox(
                label="Extra style guidance (optional)",
                placeholder="e.g., confident, friendly, avoid jargon, include call-to-action",
            )
            model_name = gr.Textbox(
                label="Gemini model",
                value="gemini-1.5-flash",
                info="Default is gemini-1.5-flash",
            )
            rewrite_btn = gr.Button("Rewrite", variant="primary")

    email_out = gr.Textbox(label="Rewritten email")
    status = gr.Markdown(visible=True)

    rewrite_btn.click(
        rewrite,
        inputs=[email_in, tone, word_limit, include_subject, extra_style, model_name],
        outputs=[email_out, status],
    )


In [25]:
demo.launch()


* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.
